# APF3 — Pipeline de Machine Learning
## Puku Puku CRM · Innovación y Transformación Digital · UTP 2026-1

**Equipo:** Adams Saldivar, Asencio Bravo, Ramirez Mendez, Ramos Atuncar  
**Random state:** 42 (reproducible)  
**Dependencias:** `pip install -r requirements.txt`

In [ ]:
import io, os, sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             RocCurveDisplay)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

API_URL = 'http://localhost:4000/api/reportes/export-apf3.csv'
AUTH_URL = 'http://localhost:4000/api/auth/login'
CREDENTIALS = {'email': 'admin@pukupuku.pe', 'password': 'puku2026'}

---
## 1. Carga del dataset

In [ ]:
def cargar_dataset():
    local = Path('dataset_apf3_puku_puku.csv')
    if local.exists():
        print(f'Cargando desde archivo local: {local}')
        return pd.read_csv(local)
    try:
        r = requests.post(AUTH_URL, json=CREDENTIALS, timeout=10)
        r.raise_for_status()
        token = r.json()['token']
        r2 = requests.get(API_URL, headers={'Authorization': f'Bearer {token}'}, timeout=15)
        r2.raise_for_status()
        print(f'Cargando desde API: {API_URL}')
        return pd.read_csv(io.StringIO(r2.text))
    except Exception as e:
        print(f'API no disponible ({e}). Generando dataset simulado...')
        return _dataset_simulado()

def _dataset_simulado():
    rng = np.random.default_rng(RANDOM_STATE)
    n = 200
    return pd.DataFrame({
        'nombre': [f'Cliente_{i}' for i in range(n)],
        'frecuencia_visita': rng.poisson(lam=4, size=n).clip(0, 20),
        'ticket_promedio_soles': np.round(rng.exponential(scale=16, size=n).clip(3, 60), 2),
        'canal_origen': rng.choice(['PRESENCIAL','WHATSAPP','INSTAGRAM','RAPPI','PEDIDOSYA'], size=n),
        'producto_favorito': rng.choice(['Flat white sin azúcar','Cappuccino clásico','Latte caramel',
                                         'Matcha latte','Americano','Mocha','Chai latte'], size=n),
        'churn_label': (rng.random(n) < 0.3).astype(int),
        'churn_score': np.round(np.clip(np.random.random(n), 0, 1), 4),
    })

df = cargar_dataset()
print(f'Dataset: {df.shape[0]} filas, {df.shape[1]} columnas')
df.head(3)

---
## 2. Análisis Exploratorio (EDA)

In [ ]:
print('Nulos por columna:')
print(df.isnull().sum())
print('\nDescriptivas:')
display(df.describe().round(3))
print('\nDistribución target:')
print(df['churn_label'].value_counts())
print(f'\nChurn rate: {df["churn_label"].mean()*100:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('EDA — Puku Puku CRM | APF3', fontsize=14, fontweight='bold')

df['churn_label'].value_counts().plot(ax=axes[0,0], kind='bar', color=['#4f7942','#c1502e'])
axes[0,0].set_title('churn_label')
axes[0,0].set_xticklabels(['Activo','Churn'], rotation=0)

df['canal_origen'].value_counts().plot(ax=axes[0,1], kind='bar', color='#5b3a21')
axes[0,1].set_title('Clientes por canal')
axes[0,1].tick_params(axis='x', rotation=30)

df['producto_favorito'].value_counts().head(10).plot(ax=axes[0,2], kind='barh', color='#c98a2e')
axes[0,2].set_title('Top 10 productos')

axes[1,0].hist(df['frecuencia_visita'], bins=15, color='#4f7942', edgecolor='white')
axes[1,0].set_title('Frecuencia de visitas')

axes[1,1].hist(df['ticket_promedio_soles'], bins=15, color='#2e6b8a', edgecolor='white')
axes[1,1].set_title('Ticket promedio (S/)')

axes[1,2].hist([df[df['churn_label']==0]['frecuencia_visita'],
                df[df['churn_label']==1]['frecuencia_visita']],
               bins=12, label=['Activo','Churn'], color=['#4f7942','#c1502e'], alpha=0.7)
axes[1,2].set_title('Frecuencia por churn')
axes[1,2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
sns.heatmap(df[['frecuencia_visita','ticket_promedio_soles','churn_label']].corr(),
            annot=True, cmap='BrBG', vmin=-1, vmax=1, center=0)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlacion.png', dpi=150)
plt.show()

---
## 3. Preprocesamiento

In [ ]:
le_canal = LabelEncoder()
df['canal_enc'] = le_canal.fit_transform(df['canal_origen'])

le_prod = LabelEncoder()
df['producto_enc'] = le_prod.fit_transform(df['producto_favorito'].fillna('N/A'))

feature_cols = ['frecuencia_visita', 'ticket_promedio_soles', 'canal_enc', 'producto_enc']
X = df[feature_cols].values
y = df['churn_label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')
print(f'Clases train: {np.bincount(y_train)}')
print(f'Clases test:  {np.bincount(y_test)}')

---
## 4. Clasificación supervisada

In [ ]:
modelos = {
    'Regresión Logística': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200, max_depth=8, class_weight='balanced'),
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    
    resultados.append({
        'modelo': nombre,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'auc': roc_auc_score(y_test, y_prob),
    })
    print(f'\n  {nombre}')
    print(f'    Matriz de confusión:\n{confusion_matrix(y_test, y_pred)}')
    print(classification_report(y_test, y_pred, zero_division=0))
    
    RocCurveDisplay.from_estimator(modelo, X_test, y_test)
    plt.title(f'Curva ROC — {nombre}')
    plt.savefig(OUTPUT_DIR / f'roc_{nombre.lower().replace(" ", "_")}.png', dpi=150)
    plt.show()

df_result = pd.DataFrame(resultados).set_index('modelo')
print('\nComparación de modelos:')
display(df_result.round(4))

In [ ]:
rf = modelos['Random Forest']
imp = pd.DataFrame({'feature': feature_cols, 'importancia': rf.feature_importances_})
imp = imp.sort_values('importancia', ascending=False)

plt.figure(figsize=(7,4))
sns.barplot(data=imp, x='importancia', y='feature', palette='BrBG')
plt.title('Importancia de features — Random Forest')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=150)
plt.show()
imp

---
## 5. Segmentación no supervisada (K-Means)

In [ ]:
X_seg = df[['frecuencia_visita', 'ticket_promedio_soles']].values
X_seg = StandardScaler().fit_transform(X_seg)

inercias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_seg)
    inercias.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(range(2,9), inercias, marker='o', color='#c1502e')
plt.xlabel('k (clusters)')
plt.ylabel('Inercia')
plt.title('Método del codo — K-Means')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
k = 3
km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
df['segmento'] = km.fit_predict(X_seg)

print(f'Perfiles de segmentos (k={k}):')
for seg in range(k):
    sub = df[df['segmento'] == seg]
    print(f'\n  Segmento {seg} ({len(sub)} clientes, {len(sub)/len(df)*100:.1f}%)')
    print(f'    Frecuencia prom.:  {sub["frecuencia_visita"].mean():.2f}')
    print(f'    Ticket prom.:      S/{sub["ticket_promedio_soles"].mean():.2f}')
    print(f'    Churn rate:        {sub["churn_label"].mean()*100:.1f}%')
    print(f'    Canal predominante: {sub["canal_origen"].mode().iloc[0]}')

In [ ]:
plt.figure(figsize=(8,5))
scatter = plt.scatter(df['frecuencia_visita'], df['ticket_promedio_soles'],
                      c=df['segmento'], cmap='Set2', edgecolor='black', alpha=0.7, s=60)
plt.xlabel('Frecuencia de visitas')
plt.ylabel('Ticket promedio (S/)')
plt.title('Segmentos K-Means (k=3)')
plt.legend(handles=scatter.legend_elements()[0],
           labels=[f'Segmento {i}' for i in range(k)], title='Cluster')
plt.grid(alpha=0.3)
plt.savefig(OUTPUT_DIR / 'segmentos_scatter.png', dpi=150)
plt.show()

---
## 6. Reproducibilidad

| Parámetro | Valor |
|-----------|-------|
| random_state | 42 |
| Test size | 30% |
| Stratify | Sí |
| Scaler | StandardScaler |
| Modelos | LogisticRegression, RandomForest (class_weight='balanced') |
| K-Means | k=3, n_init=10 |

**Outputs generados en `output/`:**
- `eda.png` — Distribuciones exploratorias
- `correlacion.png` — Matriz de correlación
- `roc_regresión_logística.png`, `roc_random_forest.png` — Curvas ROC
- `comparacion_modelos.csv` — Métricas
- `feature_importance.png` — Importancia de variables
- `codo_kmeans.png`, `segmentos_scatter.png` — Segmentación
- `segmentos.csv` — Dataset con columna segmento
- `informe_reproducibilidad.txt` — Resumen ejecutable